In [1]:
import requests
from bs4 import BeautifulSoup
import webvtt
from io import StringIO

In [2]:
url = 'https://api.vlivearchive.com'
path = '/video/'

In [3]:
# insert into this list the desired video ID numbers

videoList = ['178118', '178124', '191726', '200141', '217103', '217104', '217110', '222179', '222184', '228353', '244356']

In [4]:
# temporary container will house a series of dictionaries with various video metadata

container = []

for vid in videoList:
    r = requests.get(url + path + vid)
    container.append(r.json())

In [5]:
# this loop extracts ALL video titles and, if captions exist, 
# will extract the timecode and text from the first linked 
# VTT file that matches the specified language for each video

# this loop can be modified to extract additional kinds of metadata from the called dictionaries
# this loop purposefully combines all captions of all requested videos into a single list

allSubs = []

for d in container:
    allSubs.append(d['title'])
    if 'captions' in d:
        for i in d['captions']:
            if 'en_US' in i['source']:
                url = i['source']
                payload = requests.get(url)
                buffer = StringIO(payload.text)
                for caption in webvtt.from_buffer(buffer):
                    allSubs.append(caption.start)
                    allSubs.append(caption.text)
    else:
        allSubs.append('null')

In [6]:
# this saves the list into a single text file
# this can be modified to save separate files for each item aka each video's extracted subtitles

with open('output_files/subs.txt', 'w', encoding="utf-8") as f:
    for item in allSubs:
        if str(item).strip()[:1].isdigit():
            f.write(f"\n\n{item}")
        else:
            f.write(f"\n{item}")